In [21]:
import os
file_names = os.listdir("wiki")
for i in range(3):
    print(file_names[i])
    print(len(file_names[i]))

Fortuna_Glacier.html
20
Timothy_Wanyonyi_Wetangula.html
31
Dakar_2_The_World27s_Ultimate_Rally.html
40


In [22]:
folder_name = "wiki"
file_name = "Fortuna_Glacier.html"
with open(os.path.join(folder_name, file_name)) as f:
    lines = [line for line in f.readlines()]

In [3]:
#with open(os.path.join("wiki", file_names[0])) as f:
  #  print(f.read())

Adding the Mapreducre |Framework

In [23]:
import math
import functools
from multiprocessing import Pool

def make_chunks(data, num_chunks):
    chunk_size = math.ceil(len(data) / num_chunks)
    return [data[i:i+chunk_size] for i in range(0, len(data), chunk_size)]

def map_reduce(data, num_processes, mapper, reducer):
    chunks = make_chunks(data, num_processes)
    pool = Pool(num_processes)
    chunk_results = pool.map(mapper, chunks)
    return functools.reduce(reducer, chunk_results)

Count the total number of lines in all fines

In [24]:
def map_line_count(file_names):
    total = 0
    for fn in file_names:
        with open(os.path.join("wiki", fn)) as f:
            total += len(f.readlines())
    return total

def reducer_line_count(count1,count2):
    return count1 + count2

target = map_reduce(file_names, 4, map_line_count, reducer_line_count)
print(target)

499797


Grep string function
We defined a mapreduce_grep_string() function that takes two arguments as input:

A path to a folder. In the case of this guided project we will only use it on the wiki folder but having this argument makes the function easier to reuse.

The string that we want to find.

The mapper function receives a chunk of filenames and calculates all occurrences of the target string on them. If a file contains no occurrences, we chose to not include an entry for that file in the result dictionary.

The reducer function uses the dict.update() method to merge the result dictionaries.

Note that the target variable will be defined outside and will be the string w

In [10]:
def map_grep(file_names):
    results = {}
    for fn in file_names:
        with open(fn) as f:
            lines = [line for line in f.readlines()]
        for line_index, line in enumerate(lines):
            if target in line:
                if fn not in results:
                    results[fn] = []
                results[fn].append(line_index)
    return results

def reduce_grep(lines1, lines2):
    lines1.update(lines2)
    return lines1

def mapreduce_grep(path, num_processes):
    file_names = [os.path.join(path, fn) for fn in os.listdir(path)]
    return map_reduce(file_names, num_processes,  map_grep, reduce_grep)

In [25]:
target = "data"
data_occurrences = mapreduce_grep("wiki", 8)


target = "data"
data_occurrences = mapreduce_grep("wiki", 8)
Allow for case insensitive matches
We can allow case insensitive matches by converting both the target and the file contents to lowercase before we match.

In [26]:
def map_grep_insensitive(file_names):
    results = {}
    for fn in file_names:
        with open(fn) as f:
            lines = [line.lower() for line in f.readlines()]
        for line_index, line in enumerate(lines):
            if target.lower() in line:
                if fn not in results:
                    results[fn] = []
                results[fn].append(line_index)
    return results

def mapreduce_grep_insensitive(path, num_processes):
    file_names = [os.path.join(path, fn) for fn in os.listdir(path)]
    return map_reduce(file_names, num_processes,  map_grep_insensitive, reduce_grep)

target = "data"
new_data_occurrences = mapreduce_grep_insensitive("wiki", 8)

Checking that we find more matches
We already stored the results into variables data_occurrences and new_data_occurrences. To check that we find more matches with the second version of the algorithm, we can loop over the file names and print the length difference between the results.

In [27]:
for fn in new_data_occurrences:
    if fn not in data_occurrences:
        print("Found {} new matches on file {}".format(len(new_data_occurrences[fn]), fn))
    elif len(new_data_occurrences[fn]) > len(data_occurrences[fn]):
        print("Found {} new matches on file {}".format(len(new_data_occurrences[fn]) - len(data_occurrences[fn]), fn))

Found 4 new matches on file wiki/List_of_molecular_graphics_systems.html
Found 2 new matches on file wiki/Gordon_Bau.html
Found 1 new matches on file wiki/Teiji_Ito.html
Found 1 new matches on file wiki/Frost_Township_Michigan.html
Found 1 new matches on file wiki/Sahanpur.html
Found 1 new matches on file wiki/Holly_Golightly_(comics).html
Found 1 new matches on file wiki/Mudramothiram.html
Found 2 new matches on file wiki/Taipa_HousesE28093Museum.html
Found 1 new matches on file wiki/Don_Parsons_(ice_hockey).html
Found 1 new matches on file wiki/Gulliver_Mickey.html
Found 1 new matches on file wiki/Cobble_Hill_Brooklyn.html
Found 1 new matches on file wiki/Meleh_Kabude_Sofla.html
Found 1 new matches on file wiki/CurtissWright_Hangar_(Columbia_South_Carolina).html
Found 1 new matches on file wiki/Julien_Boisselier.html
Found 1 new matches on file wiki/Morning_Glory_(2010_film).html
Found 1 new matches on file wiki/Don_Raye.html
Found 1 new matches on file wiki/Camp_Nelson_Confederate_C

Finding all match locations
We can use any of the above functions to find all match locations. We will use the third one.

After finding all indexes in one line, we need to create pairs by adding the line index.

In [29]:
def map_grep_match_indexes(file_names):
    results = {}
    for fn in file_names:
        with open(fn) as f:
            lines = [line.lower() for line in f.readlines()]
        for line_index, line in enumerate(lines):
            target_str = target.lower()
            match_indexes = []
            i = line.find(target_str)
            while i != -1:
                match_indexes.append(i)
                i = line.find(target_str, i + 1)
                
            if match_indexes:
                if fn not in results:
                    results[fn] = []
                results[fn] += [(line_index, match_index) for match_index in match_indexes]
    return results

def mapreduce_grep_match_indexes(path, num_processes):
    file_names = [os.path.join(path, fn) for fn in os.listdir(path)]
    return map_reduce(file_names, num_processes, map_grep_match_indexes, reduce_grep)

target = "science"
occurrences = mapreduce_grep_match_indexes("wiki", 8)

Displaying the results
Let's display the results. We will create a CSV file listing all occurrences. We will also show the text around each occurrence.

In [30]:
import csv

# How many character to show before and after the match
context_delta = 30

with open("results.csv", "w") as f:
    writer = csv.writer(f)
    rows = [["File", "Line", "Index", "Context"]]
    for fn in occurrences:
        with open(fn) as f:
            lines = [line.strip() for line in f.readlines()]
        for line, index in occurrences[fn]:
            start = max(index - context_delta, 0)
            end   = index + len(target) + context_delta
            rows.append([fn, line, index, lines[line][start:end]])
    writer.writerows(rows)

In [31]:
import pandas
df = pandas.read_csv("results.csv")
df.head(10)

,File,Line,Index,Context
0,wiki/List_of_molecular_graphics_systems.html,411,22,"<td>Computational nanoscience: life sciences, ..."
1,wiki/List_of_molecular_graphics_systems.html,411,36,"mputational nanoscience: life sciences, materi..."
2,wiki/List_of_molecular_graphics_systems.html,642,288,"ez"". <i>Trends in Biochemical Sciences</i>. <b..."
3,wiki/List_of_molecular_graphics_systems.html,642,1230,.jtitle=Trends+in+Biochemical+Sciences&amp;rft...
4,wiki/List_of_molecular_graphics_systems.html,659,258,"ll"". <i>Trends in Biochemical Sciences</i>. <b..."
5,wiki/List_of_molecular_graphics_systems.html,659,1131,.jtitle=Trends+in+Biochemical+Sciences&amp;rft...
6,wiki/List_of_molecular_graphics_systems.html,660,257,"ts"". <i>Trends in Biochemical Sciences</i>. <b..."
7,wiki/List_of_molecular_graphics_systems.html,660,1118,.jtitle=Trends+in+Biochemical+Sciences&amp;rft...
8,wiki/List_of_molecular_graphics_systems.html,664,241,s.pl/Chemistry_Materials_Life_Science/products...
9,wiki/List_of_molecular_graphics_systems.html,664,567,pl%2FChemistry_Materials_Life_Science%2Fproduc...
